### Introduction

### Import librairies

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.special import ndtr

# Find the project root so this notebook also works when launched from a subfolder.
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "python_module" / "pricing_model.py").is_file():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError("Could not find python_module/pricing_model.py from the notebook directory")

from python_module.pricing_model import MultiAssetBSMModel, SABRModel

### Custom functions

In [2]:
def price_and_delta(forwards, tau, strikes, ivs, is_call, rate):
    """Vectorized Black-76 marks for all paths and options at one time step.

    `strikes`, `ivs` and `is_call` are row vectors of shape (1, n_options).
    """
    forwards = np.asarray(forwards, dtype=float).reshape(-1, 1)
    if tau <= 0:
        intrinsic_call = np.maximum(forwards - strikes, 0.0)
        intrinsic_put = np.maximum(strikes - forwards, 0.0)
        intrinsic = np.where(is_call, intrinsic_call, intrinsic_put)
        # Expiry delta is not used: there is no hedge interval after maturity.
        return intrinsic, np.zeros_like(intrinsic)

    sqrt_tau = np.sqrt(tau)
    d1 = (np.log(forwards / strikes) + 0.5 * ivs**2 * tau) / (ivs * sqrt_tau)
    d2 = d1 - ivs * sqrt_tau
    discount = np.exp(-rate * tau)

    call_pv = discount * (forwards * ndtr(d1) - strikes * ndtr(d2))
    put_pv = discount * (strikes * ndtr(-d2) - forwards * ndtr(-d1))
    pv = np.where(is_call, call_pv, put_pv)
    call_delta = discount * ndtr(d1)
    put_delta = discount * (ndtr(d1) - 1.0)
    delta = np.where(is_call, call_delta, put_delta)
    return pv, delta


def option_gamma_at_spot(spot, tau, strikes, vols, rate):
    """Black-76 gammas at one evaluation level; singular expiry gammas are NaN."""
    if tau < 0:
        raise ValueError("Remaining maturity cannot be negative")
    if tau == 0:
        return np.where(strikes == spot, np.nan, 0.0)
    sqrt_tau = np.sqrt(tau)
    d1 = (
        np.log(spot / strikes) + 0.5 * vols**2 * tau
    ) / (vols * sqrt_tau)
    pdf_d1 = np.exp(-0.5 * d1**2) / np.sqrt(2.0 * np.pi)
    return np.exp(-rate * tau) * pdf_d1 / (spot * vols * sqrt_tau)


def portfolio_gamma_cash_grid(strikes, vols, weights, maturity, n_steps, rate):
    """Portfolio cash gamma (per 1% move) with the forward at each strike, at each step.

    Returns an array of shape (n_steps + 1, n_strikes). Expiry values at a strike are NaN.
    """
    active = weights != 0.0
    grid = np.empty((n_steps + 1, len(strikes)))
    for step in range(n_steps + 1):
        # Explicit zero at the final step avoids a spurious finite expiry gamma.
        tau = 0.0 if step == n_steps else maturity * (n_steps - step) / n_steps
        for j, spot in enumerate(strikes):
            gammas = option_gamma_at_spot(spot, tau, strikes, vols, rate)
            portfolio_gamma = gammas[active] @ weights[active]
            grid[step, j] = portfolio_gamma * spot**2 / 100.0
    return grid


def sabr_implied_vol(F, K, tau, alpha, beta, rho, nu):
    """Vectorized Hagan SABR implied vol; same formula as SABRModel.compute_sigma.

    `F` and `alpha` broadcast against `K` (e.g. shape (n_paths, 1) against (1, n_options)).
    """
    F, K, alpha = np.asarray(F, dtype=float), np.asarray(K, dtype=float), np.asarray(alpha, dtype=float)
    FK_beta = (F * K) ** ((1 - beta) / 2)
    log_FK = np.log(F / K)
    z = (nu / alpha) * FK_beta * log_FK
    x_z = np.log((np.sqrt(1 - 2 * rho * z + z**2) + z - rho) / (1 - rho))
    # z / x(z) -> 1 at the money, which recovers the F == K branch of compute_sigma.
    near_atm = np.abs(z) < 1e-12
    z_over_x = np.where(near_atm, 1.0, z / np.where(near_atm, 1.0, x_z))
    factor1 = alpha / (FK_beta * (1 + (1 - beta) ** 2 * log_FK**2 / 24 + (1 - beta) ** 4 * log_FK**4 / 1920))
    factor3 = 1 + (
        ((1 - beta) ** 2 * alpha**2) / (24 * F ** (2 - 2 * beta))
        + 0.25 * rho * beta * nu * alpha / F ** (1 - beta)
        + (2 - 3 * rho**2) * nu**2 / 24
    ) * tau
    return factor1 * z_over_x * factor3


def black76_marks(F, K, tau, iv, rate, is_call):
    """Vectorized Black-76 price and Greeks, in the units of BSMModel.compute_option_with_forward.

    vega is per vol point, theta per trading day. At expiry the price is intrinsic and the Greeks are NaN.
    """
    F = np.asarray(F, dtype=float)
    if tau <= 0:
        pv = np.where(is_call, np.maximum(F - K, 0.0), np.maximum(K - F, 0.0))
        nan = np.full_like(pv, np.nan)
        return {"pv": pv, "delta": nan, "gamma": nan, "vega": nan, "theta": nan, "vanna": nan, "volga": nan}

    sqrt_tau = np.sqrt(tau)
    d1 = (np.log(F / K) + 0.5 * iv**2 * tau) / (iv * sqrt_tau)
    d2 = d1 - iv * sqrt_tau
    discount = np.exp(-rate * tau)
    pdf_d1 = np.exp(-0.5 * d1**2) / np.sqrt(2.0 * np.pi)

    pv = discount * np.where(is_call, F * ndtr(d1) - K * ndtr(d2), K * ndtr(-d2) - F * ndtr(-d1))
    vega = discount * F * pdf_d1 * sqrt_tau / 100
    return {
        "pv": pv,
        "delta": discount * np.where(is_call, ndtr(d1), ndtr(d1) - 1.0),
        "gamma": discount * pdf_d1 / (F * iv * sqrt_tau),
        "vega": vega,
        "theta": (-F * pdf_d1 * iv / (2 * sqrt_tau) * discount + rate * pv) / 252,
        "vanna": -discount * d2 * pdf_d1 / iv,
        "volga": vega * 100 * d1 * d2 / iv,
    }


def sabr_option_marks(F, alpha_t, tau, strikes, is_call, beta, rho, nu, rate, bump=1e-4):
    """Re-mark options under the SABR state (F, alpha_t): IV, Black-76 price and Greeks, and SABR delta.

    sabr_delta = Black-76 delta + raw vega * dIV/dF, where dIV/dF is a central difference on the
    SABR smile with alpha held fixed (Hagan's model delta).
    bartlett_delta = sabr_delta + raw vega * dIV/dalpha * rho * nu / F**beta, which adds the
    expected move of alpha given a move of F under the SABR dynamics.
    """
    iv = sabr_implied_vol(F, strikes, tau, alpha_t, beta, rho, nu)
    marks = black76_marks(F, strikes, tau, iv, rate, is_call)
    marks["iv"] = iv
    if tau <= 0:
        marks["sabr_delta"] = np.full_like(marks["pv"], np.nan)
        marks["bartlett_delta"] = np.full_like(marks["pv"], np.nan)
        return marks
    dF = bump * np.asarray(F, dtype=float)
    div_dF = (
        sabr_implied_vol(F + dF, strikes, tau, alpha_t, beta, rho, nu)
        - sabr_implied_vol(F - dF, strikes, tau, alpha_t, beta, rho, nu)
    ) / (2 * dF)
    marks["sabr_delta"] = marks["delta"] + marks["vega"] * 100 * div_dF
    d_alpha = bump * np.asarray(alpha_t, dtype=float)
    div_dalpha = (
        sabr_implied_vol(F, strikes, tau, alpha_t + d_alpha, beta, rho, nu)
        - sabr_implied_vol(F, strikes, tau, alpha_t - d_alpha, beta, rho, nu)
    ) / (2 * d_alpha)
    marks["bartlett_delta"] = marks["sabr_delta"] + marks["vega"] * 100 * div_dalpha * rho * nu / np.asarray(F, dtype=float) ** beta
    return marks

### Inputs

In [3]:
# Universe inputs
n_options = 100
lower_delta = -0.001         # Signed forward delta for the lower-wing put (-0.1%)
upper_delta = 0.001          # Forward delta for the upper-wing call (+0.1%)
maturity_years = 1 / 12      # One month

# Initial market and standard equity SABR inputs
F0 = 100.0                   # Initial forward
risk_free_rate = 0.04
alpha = 0.20                 # With beta=1, alpha is the initial volatility level
beta = 1.00
rho = -0.90
nu = -0.9

# Monte Carlo inputs
n_paths = 5000
n_steps = max(1, round(maturity_years * 252))
random_seed = 44
hedge_delta = "sabr_delta"   # Delta used for the daily forward hedge: "sabr_delta" (model) or "delta" (Black-76)

### 1. Define a sample portfolio


In [4]:
if n_options < 2:
    raise ValueError("n_options must be at least 2 to include both delta wings")
if not (-0.5 < lower_delta < 0 < upper_delta < 0.5):
    raise ValueError("Use an OTM put delta below zero and an OTM call delta above zero")
if maturity_years <= 0:
    raise ValueError("maturity_years must be positive")

print(f"Options: {n_options} | maturity: {maturity_years:.6f} years | steps: {n_steps} | paths: {n_paths}")


put_wing_strike = SABRModel.solve_delta_strike(
    F=F0, T=maturity_years, alpha=alpha, beta=beta, rho=rho, nu=nu,
    r=risk_free_rate, option_type="put", target_delta=lower_delta,
)
call_wing_strike = SABRModel.solve_delta_strike(
    F=F0, T=maturity_years, alpha=alpha, beta=beta, rho=rho, nu=nu,
    r=risk_free_rate, option_type="call", target_delta=upper_delta,
)
if not put_wing_strike < F0 < call_wing_strike:
    raise ValueError("Solved wing strikes did not bracket the initial forward")

strikes = np.linspace(put_wing_strike, call_wing_strike, n_options)
option_types = np.where(strikes < F0, "put", "call")

option_rows = []
for strike, option_type in zip(strikes, option_types):
    result = SABRModel.compute_option(
        F=F0, K=float(strike), T=maturity_years,
        alpha=alpha, beta=beta, rho=rho, nu=nu,
        r=risk_free_rate, option_type=str(option_type),
        compute_bs_greeks=True, compute_model_greek=False,
    )
    option_rows.append({
        "strike": strike,
        "option_type": option_type,
        "IV": result["IV"],
        "pv": result["price"],
        "delta": result["delta"],
        "gamma": result["gamma"],
        "vega_per_vol_point": result["vega"],
        "theta_per_day": result["theta"],
    })

option_universe = pd.DataFrame(option_rows)
# Variance swap replication (midpoint rule): w_i = dK_i / K_i**2, where dK_i is the width of the
# strike interval centred on K_i, i.e. (K_{i+1} - K_{i-1}) / 2 inside the grid and one full step at the wings.
option_universe["dK"] = np.gradient(option_universe["strike"].to_numpy())
option_universe["raw_weight"] = option_universe["dK"] / option_universe["strike"] ** 2
option_universe["weight"] = option_universe["raw_weight"] / option_universe["raw_weight"].sum()

portfolio_initial = {
    "pv": float(option_universe["weight"] @ option_universe["pv"]),
    "delta": float(option_universe["weight"] @ option_universe["delta"]),
    "gamma": float(option_universe["weight"] @ option_universe["gamma"]),
    "vega_per_vol_point": float(option_universe["weight"] @ option_universe["vega_per_vol_point"]),
    "theta_per_day": float(option_universe["weight"] @ option_universe["theta_per_day"]),
}

Options: 100 | maturity: 0.083333 years | steps: 21 | paths: 5000


In [5]:
import plotly.graph_objects as go

gamma_cash_steps = np.arange(n_steps + 1)
gamma_cash_by_step = portfolio_gamma_cash_grid(
    strikes=option_universe["strike"].to_numpy(dtype=float),
    vols=option_universe["IV"].to_numpy(dtype=float),
    weights=option_universe["weight"].to_numpy(dtype=float),
    maturity=maturity_years,
    n_steps=n_steps,
    rate=risk_free_rate,
)

strike_values = option_universe["strike"].to_numpy(dtype=float)
delta_labels = [f"{100 * d:+.1f}Δ" for d in option_universe["delta"]]
hover_labels = np.tile(delta_labels, (len(gamma_cash_steps), 1))

# Label about a dozen evenly spaced strikes on the y axis with their delta.
option_deltas = option_universe["delta"].to_numpy()
n_ticks = 12
tick_idx = np.unique(np.linspace(0, len(strike_values) - 1, n_ticks).round().astype(int))
tick_vals = strike_values[tick_idx]
tick_text = [f"{100 * option_deltas[i]:+.1f}Δ (K={strike_values[i]:.1f})" for i in tick_idx]

gamma_step_fig = go.Figure(
    go.Heatmap(
        x=gamma_cash_steps,
        y=strike_values,
        z=gamma_cash_by_step.T,
        customdata=hover_labels.T,
        colorscale="Viridis",
        colorbar={"title": "Cash gamma<br>per 1%"},
        hovertemplate=(
            "Step: %{x}<br>Delta: %{customdata}<br>Strike: %{y:.2f}<br>"
            "Cash gamma: %{z:.6f}<extra></extra>"
        ),
    )
)
gamma_step_fig.update_layout(
    title="Portfolio cash gamma by step and strike delta",
    template="plotly_white",
    height=800,
    xaxis={"title": "Step", "dtick": 1},
    yaxis={
        "title": "Strike (initial forward delta)",
        "tickmode": "array",
        "tickvals": tick_vals,
        "ticktext": tick_text,
    },
)
gamma_step_fig.show()

/opt/anaconda3/lib/python3.12/site-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.




### 2. Generate a Monte Carlo simulation 

Simulate the forward and the SABR volatility state with `SABRModel.compute_montecarlo` (Euler scheme, same SABR inputs as above). At every step and on every path, each option is re-marked under the SABR dynamic: its implied vol is recomputed from the current forward `F_t`, the current volatility state `alpha_t` and the remaining maturity, then priced with Black-76.

For each option, P&L over step `t-1 -> t` (per unit of option):

```text
option_pnl       = pv_t - pv_{t-1}
funding_pnl      = -pv_{t-1} * (exp(r * dt) - 1)          premium financed at the risk-free rate
hedge_pnl        = -hedge_delta_{t-1} * (F_t - F_{t-1})     forward hedge, rebalanced every step
delta_hedged_pnl = option_pnl + funding_pnl + hedge_pnl
```

Outputs:
- `sabr_mc[metric]`: per-option array of shape `(n_steps + 1, n_paths, n_options)` for `iv, pv, delta, sabr_delta, gamma, vega, theta, vanna, volga, option_pnl, hedge_pnl, delta_hedged_pnl` (stored as float32 to save memory; P&L at step 0 is zero, Greeks at expiry are NaN).
- `portfolio_mc[metric]`: weighted portfolio value of each metric, a DataFrame indexed by step with one column per path (float64).

In [6]:
# Simulate forward and SABR volatility paths with the library model: DataFrames (n_steps + 1, n_paths).
F_df, alpha_df = SABRModel.compute_montecarlo(
    F=F0, T=maturity_years, alpha=alpha, beta=beta, rho=rho, nu=nu,
    n_steps=n_steps, n_paths=n_paths, seed=True, seed_value=random_seed,
)
forward_paths = F_df.to_numpy()
alpha_paths = alpha_df.to_numpy()
dt = maturity_years / n_steps

mc_strikes = option_universe["strike"].to_numpy(dtype=float)[None, :]
mc_is_call = (option_universe["option_type"].to_numpy() == "call")[None, :]
mc_weights = option_universe["weight"].to_numpy(dtype=float)

mark_metrics = ["iv", "pv", "delta", "sabr_delta", "gamma", "vega", "theta", "vanna", "volga"]
pnl_metrics = ["option_pnl", "hedge_pnl", "delta_hedged_pnl"]
cube_shape = (n_steps + 1, n_paths, n_options)
sabr_mc = {metric: np.zeros(cube_shape, dtype=np.float32) for metric in mark_metrics + pnl_metrics}
portfolio_arrays = {metric: np.zeros((n_steps + 1, n_paths)) for metric in mark_metrics + pnl_metrics}

previous = None
for step in range(n_steps + 1):
    tau = maturity_years * (n_steps - step) / n_steps
    marks = sabr_option_marks(
        forward_paths[step][:, None], alpha_paths[step][:, None], tau,
        mc_strikes, mc_is_call, beta, rho, nu, risk_free_rate,
    )
    if previous is not None:
        option_pnl = marks["pv"] - previous["pv"]
        funding_pnl = -previous["pv"] * np.expm1(risk_free_rate * dt)
        hedge_pnl = -previous[hedge_delta] * (forward_paths[step] - forward_paths[step - 1])[:, None]
        marks.update({
            "option_pnl": option_pnl,
            "hedge_pnl": hedge_pnl,
            "delta_hedged_pnl": option_pnl + funding_pnl + hedge_pnl,
        })
    for metric in sabr_mc:
        if metric in marks:
            sabr_mc[metric][step] = marks[metric]
            # IV of a portfolio is not additive, so only the other metrics are aggregated.
            if metric != "iv":
                portfolio_arrays[metric][step] = marks[metric] @ mc_weights
    previous = marks

portfolio_mc = {
    metric: pd.DataFrame(values, index=pd.RangeIndex(n_steps + 1, name="step"))
    for metric, values in portfolio_arrays.items() if metric != "iv"
}
portfolio_cum_hedged_pnl = portfolio_mc["delta_hedged_pnl"].cumsum()

# Spot-check the vectorized marks against the library pricer on a few random (step, path, option) points.
rng = np.random.default_rng(0)
for step, path, option in zip(rng.integers(0, n_steps, 5), rng.integers(0, n_paths, 5), rng.integers(0, n_options, 5)):
    ref = SABRModel.compute_option(
        F=forward_paths[step, path], K=float(mc_strikes[0, option]),
        T=maturity_years * (n_steps - step) / n_steps,
        alpha=alpha_paths[step, path], beta=beta, rho=rho, nu=nu, r=risk_free_rate,
        option_type=option_universe["option_type"].iloc[option],
        compute_bs_greeks=True, compute_model_greek=True,
    )
    for metric, ref_value in (("iv", ref["IV"]), ("pv", ref["price"]), ("delta", ref["delta"]),
                              ("gamma", ref["gamma"]), ("vega", ref["vega"]), ("sabr_delta", ref["sabr_delta"])):
        assert np.isclose(sabr_mc[metric][step, path, option], ref_value, rtol=1e-3, atol=1e-5), (metric, step, path, option)

print(f"SABR Monte Carlo: {n_paths} paths x {n_steps} steps x {n_options} options | hedge with {hedge_delta}")
print(f"Per-option arrays: {len(sabr_mc)} metrics, {sum(a.nbytes for a in sabr_mc.values()) / 2**20:.0f} MB")

terminal_hedged_pnl = portfolio_cum_hedged_pnl.iloc[-1]
display(pd.DataFrame({
    "cum_option_pnl": portfolio_mc["option_pnl"].sum(),
    "cum_hedge_pnl": portfolio_mc["hedge_pnl"].sum(),
    "cum_delta_hedged_pnl": terminal_hedged_pnl,
}).describe(percentiles=[0.05, 0.5, 0.95]))

SABR Monte Carlo: 5000 paths x 21 steps x 100 options | hedge with sabr_delta
Per-option arrays: 12 metrics, 504 MB


,cum_option_pnl,cum_hedge_pnl,cum_delta_hedged_pnl
count,5000.000000,5000.000000,5000.000000
mean,0.007282,-0.007376,-0.001840
std,0.884083,0.815126,0.239924
min,-0.522414,-12.784421,-0.419489
5%,-0.520059,-1.370615,-0.278998
50%,-0.298265,0.210384,-0.054450
95%,1.491739,0.690942,0.448970
max,13.621258,1.504894,1.968526


#### Annexe1: Delta hedging

**Pricing under SABR.** Each option is priced with Black-76 at the SABR implied vol, which depends on the forward and on the SABR vol state $\alpha$:

$$
V(F, \alpha) = \text{Black76}\big(F,\ K,\ \tau,\ r,\ \sigma_{\text{SABR}}(F, K, \tau;\ \alpha, \beta, \rho, \nu)\big)
$$

$\sigma_{\text{SABR}}$ is Hagan's formula (`sabr_implied_vol`, same as `SABRModel.compute_sigma`):

$$
\sigma_{\text{SABR}} = \frac{\alpha}{(FK)^{\frac{1-\beta}{2}}\left[1 + \frac{(1-\beta)^2}{24}\ln^2\frac{F}{K} + \frac{(1-\beta)^4}{1920}\ln^4\frac{F}{K}\right]}
\cdot \frac{z}{x(z)}
\cdot \left[1 + \left(\frac{(1-\beta)^2\alpha^2}{24\,F^{2-2\beta}} + \frac{\rho\beta\nu\alpha}{4\,F^{1-\beta}} + \frac{2-3\rho^2}{24}\nu^2\right)\tau\right]
$$

$$
z = \frac{\nu}{\alpha}(FK)^{\frac{1-\beta}{2}}\ln\frac{F}{K},
\qquad
x(z) = \ln\frac{\sqrt{1-2\rho z+z^2}+z-\rho}{1-\rho},
\qquad
\frac{z}{x(z)} \to 1 \text{ at the money}
$$

**Black-76 delta.** The derivative of the Black-76 price with the implied vol held constant (`delta`):

$$
\Delta_{\text{B76}} = \frac{\partial V}{\partial F}\bigg|_{\sigma} =
\begin{cases} e^{-r\tau}\,N(d_1) & \text{call} \\ e^{-r\tau}\,\big(N(d_1)-1\big) & \text{put} \end{cases}
\qquad
d_1 = \frac{\ln(F/K) + \tfrac12\sigma^2\tau}{\sigma\sqrt{\tau}}
$$

**SABR (Hagan) delta.** The derivative of the price with $\alpha$ held constant (`sabr_delta`). When the forward moves, the implied vol of a fixed strike moves along the SABR smile, which adds a vega term:

$$
\Delta_{\text{SABR}} = \frac{\partial V}{\partial F}\bigg|_{\alpha}
= \Delta_{\text{B76}} + \mathcal{V}\,\frac{\partial \sigma_{\text{SABR}}}{\partial F}
\qquad
\mathcal{V} = \frac{\partial V}{\partial \sigma} = e^{-r\tau} F\,\varphi(d_1)\sqrt{\tau}
$$

**Computation** (`sabr_option_marks`, vectorized over paths and options at each step):
1. Compute $\sigma_{\text{SABR}}$ from the path's current $F_t$ and $\alpha_t$ and the remaining maturity $\tau$, then $\Delta_{\text{B76}}$ and vega with `black76_marks`. The `vega` output is per vol point, so $\mathcal{V} = 100 \times$ `vega`.
2. Smile slope along the forward by central difference, bumping only $F$ ($K$, $\tau$ and $\alpha$ unchanged), with $h = 10^{-4} F$ (error of order $h^2$):
$$
\frac{\partial \sigma_{\text{SABR}}}{\partial F} \approx \frac{\sigma_{\text{SABR}}(F+h) - \sigma_{\text{SABR}}(F-h)}{2h}
$$
3. `sabr_delta = delta + 100 * vega * dIV/dF`. Portfolio delta: $\Delta_{\text{port}} = \sum_i w_i \Delta_i$.
4. At expiry both deltas are NaN. They are never used there, because the last hedge is set at step `n_steps - 1`.

The spot check in the Monte Carlo cell compares `sabr_delta` with `SABRModel.compute_option(..., compute_model_greek=True)["sabr_delta"]`. That function takes a numerical derivative of the full SABR price with respect to $F$.

**Sign and shape.** With $\beta = 1$ the smile depends on moneyness $\ln(F/K)$, so it moves with the forward: $\partial\sigma/\partial F \approx -(K/F)\,\partial\sigma/\partial K$. With the equity skew ($\rho < 0$), $\partial\sigma/\partial K < 0$, so the correction is positive for every option. It follows vega × skew, which is why it peaks just below the money and fades in the wings and toward expiry (plot panels 1 and 2).

**What it leaves out (Bartlett).** $\Delta_{\text{SABR}}$ assumes $\alpha$ stays fixed when $F$ moves. Under the SABR dynamics the two are correlated ($\rho$), and the expected vol move per unit of forward move is $\rho\nu/F^{\beta}$. Bartlett's delta adds this term:

$$
\Delta_{\text{Bartlett}} = \Delta_{\text{SABR}} + \mathcal{V}\,\frac{\partial \sigma_{\text{SABR}}}{\partial \alpha}\,\frac{\rho\,\nu}{F^{\beta}}
$$


**Plot.** The four panels below show $|\Delta_{\text{B76}}|$, $|\Delta_{\text{SABR}}|$ and $|\Delta_{\text{Bartlett}}|$ against strike across the portfolio's strike range (OTM puts below $F_0$, calls above), at the initial state $F_0$, $\alpha_0$. Each panel is one remaining maturity, in trading days: the full maturity $T_0$, 75% of it, 50% of it, and the last Monte Carlo step before expiry.

In [8]:
from plotly.subplots import make_subplots

# Deltas at the initial state (F0, initial alpha) across the portfolio strikes, at four remaining maturities.
# Portfolio strikes plus F0 itself, so the at-the-money point is computed exactly.
delta_2d_strikes = np.union1d(option_universe["strike"].to_numpy(dtype=float), F0)
delta_2d_is_call = delta_2d_strikes >= F0            # OTM convention: puts below F0, calls from F0 up
delta_2d_maturities = {
    "T0": maturity_years,
    "T0 - 25%": 0.75 * maturity_years,
    "T0 - 50%": 0.50 * maturity_years,
    "Last step before expiry": maturity_years / n_steps,
}
delta_2d_styles = (
    ("delta", "Black-76 delta", "royalblue"),
    ("sabr_delta", "SABR delta", "firebrick"),
    ("bartlett_delta", "Bartlett delta", "seagreen"),
)

delta_2d_fig = make_subplots(
    rows=2, cols=2, shared_yaxes=True, vertical_spacing=0.14, horizontal_spacing=0.06,
    subplot_titles=[f"{label}: {tau * 252:.1f} trading days" for label, tau in delta_2d_maturities.items()],
)
# One legend per panel. SABR and Bartlett entries show their largest signed delta gap to Black-76 in that panel
# (signed, not on |Delta|, so the sign is the same for puts and calls: SABR - B76 = vega * dIV/dF).
delta_2d_legends = {}
for panel, tau in enumerate(delta_2d_maturities.values()):
    row, col = divmod(panel, 2)
    legend_id = "legend" if panel == 0 else f"legend{panel + 1}"
    marks = sabr_option_marks(F0, alpha, tau, delta_2d_strikes, delta_2d_is_call, beta, rho, nu, risk_free_rate)
    for name, label, color in delta_2d_styles:
        diff = marks[name] - marks["delta"]
        if name == "delta":
            legend_name = label
        else:
            worst = int(np.argmax(np.abs(diff)))
            legend_name = f"{label} (max Δ - Δ_B76: {diff[worst]:+.4f} at K={delta_2d_strikes[worst]:.1f})"
        delta_2d_fig.add_trace(go.Scatter(
            x=delta_2d_strikes, y=np.abs(marks[name]), customdata=diff, mode="lines", name=legend_name,
            line={"color": color, "width": 1.5}, legend=legend_id,
            hovertemplate=f"{label}<br>|Delta|: %{{y:.4f}}<br>Δ - Δ_B76: %{{customdata:+.4f}}<extra></extra>",
        ), row=row + 1, col=col + 1)
    delta_2d_fig.add_vline(x=F0, line={"dash": "dot", "color": "lightgray"}, row=row + 1, col=col + 1)

    x_axis = delta_2d_fig.layout["xaxis" if panel == 0 else f"xaxis{panel + 1}"]
    y_axis = delta_2d_fig.layout["yaxis" if panel == 0 else f"yaxis{panel + 1}"]
    delta_2d_legends[legend_id] = {
        "x": x_axis.domain[0] + 0.005, "y": y_axis.domain[1] - 0.005, "xanchor": "left", "yanchor": "top",
        "bgcolor": "rgba(255,255,255,0.8)", "font": {"size": 10},
    }

delta_2d_fig.update_xaxes(title_text="Strike", row=2)
# Headroom above the ATM peak so the in-panel legends do not hide the curves.
delta_2d_y_max = 1.5 * max(np.nanmax(trace.y) for trace in delta_2d_fig.data)
delta_2d_fig.update_yaxes(range=[0, delta_2d_y_max])
delta_2d_fig.update_yaxes(title_text="|Delta|", col=1)
delta_2d_fig.update_layout(
    title=f"|Delta| by strike: Black-76 vs SABR vs Bartlett (F0={F0}, alpha={alpha}, beta={beta}, rho={rho}, nu={nu})",
    template="plotly_white", height=800, hovermode="x unified",
    **delta_2d_legends,
)
delta_2d_fig.show()